# PD vs HC — Mamba training (Deliverable 3)
Pulls the latest code from GitHub and trains on the Kaggle T4 GPU.
Data comes from the attached private dataset `pd-gait-vgrf-windows`.

**Fast iteration:** after editing code locally and `git push`, just re-run the *Pull* cell and then the *Train* cell — no reinstall, no restart.

In [ ]:
# 1) GPU check + pull latest code + locate the mounted dataset
import os, subprocess, torch
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2'
cap = torch.cuda.get_device_capability()
assert cap >= (7, 5), f'Need T4 (sm_75+), got sm_{cap[0]}{cap[1]} — choose GPU T4 x2, NOT P100'
print(torch.cuda.get_device_name(0), '| torch', torch.__version__, '| cuda', torch.version.cuda)

REPO = 'https://github.com/Ahmadrezanourozii/Project-Time-Series.git'
if os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, '/kaggle/working/repo'], check=True)
print(subprocess.run(['git', '-C', '/kaggle/working/repo', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)

# Kaggle mounts datasets at varying depths (/kaggle/input/<slug> or
# /kaggle/input/datasets/<user>/<slug>) — find the npz wherever it is.
DATA = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'raw_windows_v2.npz' in files:
        DATA = root
        break
assert DATA, f"raw_windows_v2.npz not found anywhere under /kaggle/input: {[r for r, d, f in os.walk('/kaggle/input')]}"
print('DATA =', DATA, '->', sorted(os.listdir(DATA)))

In [ ]:
# 2) Install mamba-ssm (idempotent — skipped when already importable)
# Verified on this image (torch 2.10.0+cu128): latest release installs prebuilt
# wheels (mamba-ssm 2.3.2.post1 + causal-conv1d 1.6.2.post1); the old 2.2.2 pin
# has no matching wheel and tries a slow source build, so latest goes first.
import importlib.util, subprocess, sys

def try_install(*specs):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', *specs]
    print('$', ' '.join(cmd))
    return subprocess.run(cmd).returncode == 0

if importlib.util.find_spec('mamba_ssm') is None:
    ok = try_install('causal-conv1d', 'mamba-ssm')
    if not ok or importlib.util.find_spec('mamba_ssm') is None:
        print('latest failed — trying pinned 2.2.2')
        ok = try_install('causal-conv1d>=1.4.0', 'mamba-ssm==2.2.2')
    assert ok and importlib.util.find_spec('mamba_ssm') is not None, 'mamba-ssm install failed — see log'
import mamba_ssm
print('mamba_ssm', mamba_ssm.__version__, 'OK')

In [ ]:
# 3) SMOKE GATE — run this once per fresh session before any full run (~2-3 min)
!cd /kaggle/working/repo && PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0 python run_mamba.py --smoke --require-cuda-kernels \
    --npz {DATA}/raw_windows_v2.npz \
    --fold-json {DATA}/fold_assignments.json \
    --output-dir /kaggle/working/smoke

In [ ]:
# 4) FINAL — the configuration chosen by VALIDATION AUC across all sweeps
#    (60 s amplitude-statistic tokens, base encoder), on all three foot
#    configurations, plus the deployable inference bundle.
#    Choosing on validation instead of on pooled test removes the selection
#    bias that the leakage audit flagged.
import subprocess, json, pathlib

BASE = ('cd /kaggle/working/repo && PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0 '
        'python run_mamba.py --require-cuda-kernels '
        f'--fold-json {DATA}/fold_assignments.json --variant mamba2 '
        '--dropout-grid 0.1,0.3 --lr-grid 3e-4,1e-4 --collapse-retries 3')

print('=== validation-selected model, all feet, 3 seeds, + bundle ===', flush=True)
subprocess.run(f'{BASE} --npz {DATA}/stat_windows_60s.npz --model-size base --foot all '
               f'--seeds 42,43,44 --output-dir /kaggle/working/valsel_stat60_base '
               f'--export-bundle /kaggle/working/bundle', shell=True)

print('=== ablation: directionality at matched size ===', flush=True)
for tag, extra in [('bi', ''), ('uni', '--unidirectional')]:
    subprocess.run(f'{BASE} --npz {DATA}/stat_windows_60s.npz --model-size base --foot both '
                   f'--seeds 42,43,44 --output-dir /kaggle/working/abl_{tag} {extra}', shell=True)

for p in sorted(pathlib.Path('/kaggle/working').glob('valsel_*')) + \
         sorted(pathlib.Path('/kaggle/working').glob('abl_*')):
    f = p / 'results.json'
    if not f.exists(): continue
    r = json.loads(f.read_text()); a = r['args']
    for foot, blk in r['results'].items():
        s = blk['subject_mean_prob']['point']; w = blk['window']['point']
        print(f"{p.name:22s} {foot:6s} bi={str(a['bidirectional']):5s} "
              f"subj acc={s['accuracy']:.3f} f1={s['f1_w']:.3f} auc={s['auc']:.3f} | "
              f"win acc={w['accuracy']:.3f}")
print('\nbundle:', sorted(x.name for x in pathlib.Path('/kaggle/working/bundle').glob('*')))

In [ ]:
# 5) Package results for download
import json, shutil, pathlib
for p in sorted(pathlib.Path('/kaggle/working').glob('outputs*')):
    res = p / 'results.json'
    if res.exists():
        r = json.loads(res.read_text())
        for foot, blocks in r['results'].items():
            s = blocks['subject_mean_prob']['point']
            w = blocks['window']['point']
            print(f"{p.name} [{foot}] window acc={w['accuracy']} auc={w['auc']} | subject: "
                  f"acc={s['accuracy']} prec_w={s['precision_w']} rec_w={s['recall_w']} "
                  f"f1_w={s['f1_w']} auc={s['auc']}")
        shutil.make_archive(f'/kaggle/working/{p.name}_results', 'zip', str(p))